In [1]:
import pandas as pd
import numpy as np
import glob

In [4]:
import geopandas as gpd

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [28]:
RAW_DATA_PATH = "data/"

In [29]:
import os
os.listdir(RAW_DATA_PATH)

['data-2019-09.csv',
 'data-2019-10.csv',
 'data-2019-11.csv',
 'data-2019-12.csv',
 'data-2020-01.csv',
 'data-2020-02.csv',
 'data-2020-03.csv',
 'data-2020-04.csv',
 'data-2020-05.csv',
 'data-2020-06.csv',
 'data-2020-07.csv',
 'data-2020-08.csv',
 'data-2020-09.csv',
 'data-2020-10.csv',
 'data-2020-11.csv',
 'data-2020-12.csv',
 'data-2021-01.csv',
 'data-2021-02.csv',
 'data-2021-03.csv',
 'data-2021-04.csv',
 'data-2021-05.csv',
 'data-2021-06.csv',
 'data-2021-07.csv',
 'data-2021-08.csv',
 'data-2021-09.csv',
 'data-2021-10.csv',
 'data-2021-11.csv',
 'data-2021-12.csv',
 'data-2022-01.csv',
 'data-2022-02.csv',
 'data-2022-03.csv',
 'data-2022-04.csv',
 'data-2022-05.csv',
 'data-2022-07.csv',
 'data-2022-08.csv',
 'data-2022-09.csv',
 'data-2022-10.csv',
 'data-2022-11.csv',
 'data-2022-12.csv',
 'data-2023-01.csv',
 'data-2023-02.csv',
 'data-2023-03.csv',
 'data-2023-04.csv',
 'data-2023-05.csv',
 'data-2023-06.csv',
 'data-2023-07.csv',
 'data-2023-08.csv',
 'data-2023-0

In [30]:
# LOAD SENSOR METADATA

site_columns = [
    "sensor_id",
    "site_nr",
    "longitude",
    "latitude",
    "name",
    "domain",
    "road_number",
    "district",
    "municipality",
    "interval",
    "installation_date"
]

sites = pd.read_csv(
    RAW_DATA_PATH + "/sites.csv",
    header=None,
    names=site_columns
)

sites.head()

,sensor_id,site_nr,longitude,latitude,name,domain,road_number,district,municipality,interval,installation_date
0,1,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22
1,2,100052862,4.471690,51.275120,Brasschaat 2,Vlaamse Overheid A. Wegen enVerkeer,N0010002,AWV123,Brasschaat,15,2019-08-22
2,3,100052863,4.472220,51.275030,Brasschaat 1,Vlaamse Overheid A. Wegen enVerkeer,N0010001,AWV123,Brasschaat,15,2019-08-22
3,4,100052864,5.190110,51.160230,Balen 1,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,2019-08-22
4,5,100052865,5.190030,51.160180,Balen 2,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,2019-08-22


In [31]:
#  LOAD DIRECTIONS

direction_columns = [
    "sensor_id",
    "direction",
    "direction_name"]

directions = pd.read_csv(
    RAW_DATA_PATH + "/richtingen.csv",
    header=None,
    names=direction_columns)

directions.head()

,sensor_id,direction,direction_name
0,1,IN,Machelen Cyclists rich. Brucargo
1,1,OUT,Machelen Cyclists richting Machelen
2,2,IN,Brasschaat 2 Fietsers rich Merksem
3,2,OUT,Brasschaat 2 Fietsers rich Brasschaat
4,3,IN,Brasschaat 1 Fietsers rich Merksem


In [32]:
# LOAD AND AGGREGATE TRAFFIC DATA

traffic_columns = [
    "sensor_id",
    "direction",
    "vehicle_type",
    "start_time",
    "end_time",
    "count"
]

traffic_files = sorted(glob.glob(RAW_DATA_PATH + "/data-*.csv"))

daily_parts = []

for file in traffic_files:
    temp = pd.read_csv(
        file,
        header=None,
        names=traffic_columns)
    
    temp["start_time"] = pd.to_datetime(temp["start_time"])
    temp["date"] = temp["start_time"].dt.date
    temp = temp.dropna(subset=["count"])
    
    daily_temp = temp.groupby(
        ["sensor_id", "date"]
    )["count"].sum().reset_index()
    
    daily_parts.append(daily_temp)

sensor_daily = pd.concat(daily_parts, ignore_index=True)

sensor_daily.head()

,sensor_id,date,count
0,1,2019-09-01,818.0
1,1,2019-09-02,537.0
2,1,2019-09-03,601.0
3,1,2019-09-04,415.0
4,1,2019-09-05,410.0


In [33]:
# MERGE SENSOR METADATA
sensor_daily = sensor_daily.merge(
    sites[[
        "sensor_id",
        "municipality",
        "longitude",
        "latitude",
        "name"]],
    on="sensor_id",
    how="left"
)

sensor_daily.head()

,sensor_id,date,count,municipality,longitude,latitude,name
0,1,2019-09-01,818.0,Machelen,4.456122,50.916183,Machelen
1,1,2019-09-02,537.0,Machelen,4.456122,50.916183,Machelen
2,1,2019-09-03,601.0,Machelen,4.456122,50.916183,Machelen
3,1,2019-09-04,415.0,Machelen,4.456122,50.916183,Machelen
4,1,2019-09-05,410.0,Machelen,4.456122,50.916183,Machelen


In [34]:
sensor_daily.isnull().sum()

sensor_id         0
date              0
count             0
municipality    145
longitude         0
latitude          0
name              0
dtype: int64

In [35]:
sensor_daily.shape

(195783, 7)

In [36]:
# MUNICIPALITY LOCATIONS FOR WEATHER API

municipality_locations = sensor_daily.groupby("municipality")[[
    "latitude",
    "longitude"]].mean().reset_index()

municipality_locations.head()

,municipality,latitude,longitude
0,Aalst,50.934558,4.015634
1,Aalter,51.105837,3.435059
2,Aarschot,50.990840,4.812590
3,Ardooie,50.967835,3.184450
4,As,51.005090,5.603890


In [37]:
# 8. LOAD WEATHER DATA FROM LOCAL HOURLY FILE INSTEAD OF OPEN-METEO API
from pathlib import Path

weather_candidates = [
    Path("transformed_data/weather_outputs/station_weather_hourly.csv"),
    Path("modern-data-analytics/alphan/transformed_data/weather_outputs/station_weather_hourly.csv"),
]

weather_path = next((p for p in weather_candidates if p.exists()), None)
if weather_path is None:
    raise FileNotFoundError("station_weather_hourly.csv not found in expected locations")

weather_hourly = pd.read_csv(weather_path)

weather_hourly["time"] = pd.to_datetime(weather_hourly["time"], utc=True)
weather_hourly["date"] = weather_hourly["time"].dt.date

weather_2023 = (
    weather_hourly.groupby(["gemeente", "date"], as_index=False)
    .agg({
        "temperature_2m": "mean",
        "precipitation": "sum",
        "wind_speed_10m": "max",
    })
    .rename(columns={
        "gemeente": "municipality",
        "temperature_2m": "temperature_2m_mean",
        "precipitation": "precipitation_sum",
        "wind_speed_10m": "windspeed_10m_max",
    })
)

# keep the old notebook contract: later cells expect a "time" column
weather_2023["time"] = pd.to_datetime(weather_2023["date"]).astype(str)

weather_2023.head()

,municipality,date,temperature_2m_mean,precipitation_sum,windspeed_10m_max,time
0,Aalst,2019-08-21,12.600000,0.0,8.640000,2019-08-21
1,Aalst,2019-08-22,17.062500,0.0,10.315115,2019-08-22
2,Aalst,2019-08-23,18.347917,0.0,11.609651,2019-08-23
3,Aalst,2019-08-24,21.358333,0.0,10.805998,2019-08-24
4,Aalst,2019-08-25,22.845833,0.0,10.446206,2019-08-25


In [38]:
weather_2023.shape

(108594, 6)

In [39]:
weather_2023["municipality"].nunique()

71

In [40]:
#merge data with sensor_daily
weather_2023["date"] = pd.to_datetime(weather_2023["time"]).dt.date
sensor_weather_daily = sensor_daily.merge(
    weather_2023,
    on=["municipality", "date"],
    how="left")
sensor_weather_daily.head()

,sensor_id,date,count,municipality,longitude,latitude,name,temperature_2m_mean,precipitation_sum,windspeed_10m_max,time
0,1,2019-09-01,818.0,Machelen,4.456122,50.916183,Machelen,16.835417,0.0,15.277749,2019-09-01
1,1,2019-09-02,537.0,Machelen,4.456122,50.916183,Machelen,15.947917,0.0,12.245294,2019-09-02
2,1,2019-09-03,601.0,Machelen,4.456122,50.916183,Machelen,16.433333,0.2,21.129885,2019-09-03
3,1,2019-09-04,415.0,Machelen,4.456122,50.916183,Machelen,16.208333,3.6,26.220753,2019-09-04
4,1,2019-09-05,410.0,Machelen,4.456122,50.916183,Machelen,13.554167,0.6,24.122686,2019-09-05


In [41]:
sensor_weather_daily[[
    "temperature_2m_mean",
    "precipitation_sum",
    "windspeed_10m_max"
]].isnull().sum()

temperature_2m_mean    834
precipitation_sum      834
windspeed_10m_max      834
dtype: int64

In [42]:
#build weather features
sensor_weather_daily["is_rainy_day"] = (
    sensor_weather_daily["precipitation_sum"] > 0)

sensor_weather_daily["is_windy_day"] = (
    sensor_weather_daily["windspeed_10m_max"] > 30)
sensor_weather_daily.head()

,sensor_id,date,count,municipality,longitude,latitude,name,temperature_2m_mean,precipitation_sum,windspeed_10m_max,time,is_rainy_day,is_windy_day
0,1,2019-09-01,818.0,Machelen,4.456122,50.916183,Machelen,16.835417,0.0,15.277749,2019-09-01,False,False
1,1,2019-09-02,537.0,Machelen,4.456122,50.916183,Machelen,15.947917,0.0,12.245294,2019-09-02,False,False
2,1,2019-09-03,601.0,Machelen,4.456122,50.916183,Machelen,16.433333,0.2,21.129885,2019-09-03,True,False
3,1,2019-09-04,415.0,Machelen,4.456122,50.916183,Machelen,16.208333,3.6,26.220753,2019-09-04,True,False
4,1,2019-09-05,410.0,Machelen,4.456122,50.916183,Machelen,13.554167,0.6,24.122686,2019-09-05,True,False


In [43]:
# SENSOR-LEVEL FEATURES
sensor_features = sensor_weather_daily.groupby("sensor_id").agg({
    "count": ["sum", "mean"],
    "temperature_2m_mean": "mean",
    "precipitation_sum": "mean",
    "windspeed_10m_max": "mean",
    "is_rainy_day": "mean",
    "is_windy_day": "mean"})

sensor_features.columns = [
    "total_traffic",
    "avg_daily_traffic",
    "avg_temperature",
    "avg_precipitation",
    "avg_wind_speed",
    "rainy_day_ratio",
    "windy_day_ratio"]

sensor_features = sensor_features.reset_index()
sensor_features.head()

,sensor_id,total_traffic,avg_daily_traffic,avg_temperature,avg_precipitation,avg_wind_speed,rainy_day_ratio,windy_day_ratio
0,1,866447.0,381.694714,11.363990,2.274360,20.235801,0.628634,0.127753
1,2,1601974.0,705.715419,11.274517,4.887467,19.789020,0.652863,0.107048
2,3,1478394.0,651.561922,11.274054,4.889625,19.789280,0.653151,0.107096
3,4,343558.0,151.347137,11.323930,4.963636,19.427879,0.637004,0.102203
4,5,376088.0,165.750551,11.323930,4.963636,19.427879,0.637285,0.102248


In [44]:
"joined" in globals(),
"accident_counts" in globals()

False

In [48]:
# LOAD AND PREPROCESS ACCIDENT DATA

accidents = pd.read_excel(
    RAW_DATA_PATH + "/OPENDATA_MAP_2017-2024.xlsx")

flanders_accidents = accidents[
    accidents["TX_RGN_COLLISION_NL"] == "Vlaams Gewest"]

bike_accidents = flanders_accidents[
    (flanders_accidents["TX_ROAD_USR_TYPE1_NL"] == "Fiets") |
    (flanders_accidents["TX_ROAD_USR_TYPE2_NL"] == "Fiets")]

bike_accidents_geo = bike_accidents.dropna(
    subset=["MS_X_COORD", "MS_Y_COORD"])

bike_accidents_geo.shape

(53524, 45)

In [49]:
# ACCIDENT GEODATAFRAME

accidents_gdf = gpd.GeoDataFrame(
    bike_accidents_geo,
    geometry=gpd.points_from_xy(
        bike_accidents_geo["MS_X_COORD"],
        bike_accidents_geo["MS_Y_COORD"]),
    crs="EPSG:31370")
accidents_gdf.head()

,DT_YEAR_COLLISION,DT_MONTH_COLLISION,DT_TIME,CD_NIS,TX_RGN_COLLISION_FR,TX_RGN_COLLISION_NL,TX_PROV_COLLISION_FR,TX_PROV_COLLISION_NL,TX_MUNTY_COLLISION_FR,TX_MUNTY_COLLISION_NL,...,CD_ROAD_USR_TYPE2,TX_ROAD_USR_TYPE2_FR,TX_ROAD_USR_TYPE2_NL,CD_COLLISION_TYPE,TX_COLLISON_TYPE_FR,TX_COLLISION_TYPE_NL,CD_OBSTACLES,TX_OBSTACLES_FR,TX_OBSTACLES_NL,geometry
1,2017,1,0,11002,Région flamande,Vlaams Gewest,Province d’Anvers,Provincie Antwerpen,Anvers,Antwerpen,...,9,Voiture,personenauto,2,Entre 2 conducteurs: Collision frontale,Tussen 2 bestuurders: Frontale botsing,0,Pas d’obstacle,Geen hindernis,POINT (152268.069 209633.121)
13,2017,1,7,11001,Région flamande,Vlaams Gewest,Province d’Anvers,Provincie Antwerpen,Aartselaar,Aartselaar,...,3,Bicyclette,Fiets,4,Entre 2 conducteurs: Par le côté (avant/derriè...,Tss. 2 best.: langs opzij (voor-/achterkant-fl...,0,Pas d’obstacle,Geen hindernis,POINT (150662.902 203118.932)
200,2017,1,7,23094,Région flamande,Vlaams Gewest,Province du Brabant flamand,Provincie Vlaams-Brabant,Zaventem,Zaventem,...,3,Bicyclette,Fiets,4,Entre 2 conducteurs: Par le côté (avant/derriè...,Tss. 2 best.: langs opzij (voor-/achterkant-fl...,0,Pas d’obstacle,Geen hindernis,POINT (157232.033 175893.359)
1221,2017,1,16,11002,Région flamande,Vlaams Gewest,Province d’Anvers,Provincie Antwerpen,Anvers,Antwerpen,...,9,Voiture,personenauto,3,Entre 2 conducteurs: Par l'arrière,Tussen 2 bestuurders: Langs achteren,0,Pas d’obstacle,Geen hindernis,POINT (154445.154 207467.044)
1222,2017,1,15,11002,Région flamande,Vlaams Gewest,Province d’Anvers,Provincie Antwerpen,Anvers,Antwerpen,...,8,Piéton,Voetganger,5,Avec un piéton,Met een voetganger,0,Pas d’obstacle,Geen hindernis,POINT (154899.366 216114.622)


In [50]:
# SENSOR GEODATAFRAME

sensor_features = sensor_features.merge(
    sites[["sensor_id", "longitude", "latitude", "municipality", "name"]],
    on="sensor_id",
    how="left")

sensors_gdf = gpd.GeoDataFrame(
    sensor_features,
    geometry=gpd.points_from_xy(
        sensor_features["longitude"],
        sensor_features["latitude"]
    ),
    crs="EPSG:4326")

sensors_gdf = sensors_gdf.to_crs("EPSG:31370")

sensors_gdf.head()

,sensor_id,total_traffic,avg_daily_traffic,avg_temperature,avg_precipitation,avg_wind_speed,rainy_day_ratio,windy_day_ratio,longitude,latitude,municipality,name,geometry
0,1,866447.0,381.694714,11.363990,2.274360,20.235801,0.628634,0.127753,4.456122,50.916183,Machelen,Machelen,POINT (156143.904 178432.685)
1,2,1601974.0,705.715419,11.274517,4.887467,19.789020,0.652863,0.107048,4.471690,51.275120,Brasschaat,Brasschaat 2,POINT (157182.957 218365.646)
2,3,1478394.0,651.561922,11.274054,4.889625,19.789280,0.653151,0.107096,4.472220,51.275030,Brasschaat,Brasschaat 1,POINT (157219.957 218355.684)
3,4,343558.0,151.347137,11.323930,4.963636,19.427879,0.637004,0.102203,5.190110,51.160230,Balen,Balen 1,POINT (207457.483 205896.897)
4,5,376088.0,165.750551,11.323930,4.963636,19.427879,0.637285,0.102248,5.190030,51.160180,Balen,Balen 2,POINT (207451.948 205891.273)


In [51]:
# SPATIAL JOIN: ACCIDENTS TO NEAREST SENSOR-500m

joined = gpd.sjoin_nearest(
    accidents_gdf,
    sensors_gdf[["sensor_id", "geometry"]],
    how="left",
    distance_col="distance_to_sensor")

nearby_accidents = joined[
    joined["distance_to_sensor"] <= 500]

nearby_accidents.shape

(1310, 49)

In [52]:
#accident count for each sensor
#RISK DATA

accident_counts = nearby_accidents.groupby(
    "sensor_id").size().reset_index(name="accident_count")

risk_data = sensor_features[[
    "sensor_id",
    "total_traffic"]].merge(
    accident_counts,
    on="sensor_id",
    how="left")

risk_data["accident_count"] = risk_data["accident_count"].fillna(0)

risk_data["risk_rate"] = (
    risk_data["accident_count"] / risk_data["total_traffic"])

risk_data.head()

,sensor_id,total_traffic,accident_count,risk_rate
0,1,866447.0,0.0,0.000000
1,2,1601974.0,5.0,0.000003
2,3,1478394.0,6.0,0.000004
3,4,343558.0,0.0,0.000000
4,5,376088.0,4.0,0.000011


In [53]:
#add risk data
# FINAL MODEL DATASET

sensor_features = sensor_features.merge(
    risk_data[[
        "sensor_id",
        "accident_count",
        "risk_rate"]],
    on="sensor_id",
    how="left")

sensor_features[[
    "accident_count",
    "risk_rate"]] = sensor_features[[
    "accident_count",
    "risk_rate"]].fillna(0)

sensor_features.head()

,sensor_id,total_traffic,avg_daily_traffic,avg_temperature,avg_precipitation,avg_wind_speed,rainy_day_ratio,windy_day_ratio,longitude,latitude,municipality,name,accident_count,risk_rate
0,1,866447.0,381.694714,11.363990,2.274360,20.235801,0.628634,0.127753,4.456122,50.916183,Machelen,Machelen,0.0,0.000000
1,2,1601974.0,705.715419,11.274517,4.887467,19.789020,0.652863,0.107048,4.471690,51.275120,Brasschaat,Brasschaat 2,5.0,0.000003
2,3,1478394.0,651.561922,11.274054,4.889625,19.789280,0.653151,0.107096,4.472220,51.275030,Brasschaat,Brasschaat 1,6.0,0.000004
3,4,343558.0,151.347137,11.323930,4.963636,19.427879,0.637004,0.102203,5.190110,51.160230,Balen,Balen 1,0.0,0.000000
4,5,376088.0,165.750551,11.323930,4.963636,19.427879,0.637285,0.102248,5.190030,51.160180,Balen,Balen 2,4.0,0.000011


In [54]:
# TARGET VARIABLE-25% of top high risk sensors
risk_threshold = sensor_features["risk_rate"].quantile(0.75)

sensor_features["high_risk"] = (
    sensor_features["risk_rate"] >= risk_threshold)

sensor_features["high_risk"].value_counts()

high_risk
False    111
True      38
Name: count, dtype: int64

In [55]:
# MACHINE LEARNING MODEL
X = sensor_features[[
    "total_traffic",
    "avg_daily_traffic",
    "avg_temperature",
    "avg_precipitation",
    "avg_wind_speed",
    "rainy_day_ratio",
    "windy_day_ratio"]]

y = sensor_features["high_risk"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y)

model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced")

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.6
              precision    recall  f1-score   support

       False       0.71      0.77      0.74        22
        True       0.17      0.12      0.14         8

    accuracy                           0.60        30
   macro avg       0.44      0.45      0.44        30
weighted avg       0.56      0.60      0.58        30

